<a href="https://colab.research.google.com/github/handdung/bigdata/blob/main/week06_5756498_%EC%9D%B4%ED%99%98%EC%A4%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6주차 과제
빅데이터프로그래밍 · 통계학과

**이름:이환준**
**학번:5756498**

## 과제 내용
은닉층이 1개인 모델과 2개인 모델을 각각 작성하고 구조 차이를 설명합니다.

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week06_학번_이름.ipynb` 로 제출합니다.


In [1]:
import torch
import torch.nn as nn
import pandas as pd

torch.manual_seed(0)


## 문제 1. 은닉층 1개 모델 (25점)
`nn.Module` 을 상속해 아래 조건의 모델을 작성하세요.

- 입력 특성 5개, 은닉층 뉴런 16개, 출력 3개
- `__init__` 과 `forward` 를 모두 구현하고 각 줄에 주석
- `print(model)` 로 구조 출력


In [2]:
# 답안
class OneHidden(nn.Module):
    def __init__(self):
        super().__init__()                 # 부모 클래스(nn.Module) 초기화
        self.fc1 = nn.Linear(5, 16)        # 입력층 → 은닉층: 특성 5개를 뉴런 16개로
        self.relu = nn.ReLU()              # 비선형 활성화 함수
        self.fc2 = nn.Linear(16, 3)        # 은닉층 → 출력층: 뉴런 16개를 출력 3개로

    def forward(self, x):
        x = self.fc1(x)                    # (N, 5) → (N, 16)
        x = self.relu(x)                   # 음수는 0으로, shape는 그대로 (N, 16)
        x = self.fc2(x)                    # (N, 16) → (N, 3)
        return x                           # 최종 출력 반환

model1 = OneHidden()
print(model1)

OneHidden(
  (fc1): Linear(in_features=5, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=3, bias=True)
)


## 문제 2. 은닉층 2개 모델 (25점)
문제 1과 같은 입력·출력에 은닉층을 2개로 늘린 모델을 작성하고 구조를 출력하세요.


In [3]:
# 답안
class TwoHidden(nn.Module):
    def __init__(self):
        super().__init__()                 # 부모 클래스 초기화
        self.fc1 = nn.Linear(5, 16)        # 입력 5 → 은닉층1 16
        self.relu1 = nn.ReLU()             # 은닉층1 활성화
        self.fc2 = nn.Linear(16, 16)       # 은닉층1 16 → 은닉층2 16
        self.relu2 = nn.ReLU()             # 은닉층2 활성화
        self.fc3 = nn.Linear(16, 3)        # 은닉층2 16 → 출력 3

    def forward(self, x):
        x = self.relu1(self.fc1(x))        # (N, 5) → (N, 16)
        x = self.relu2(self.fc2(x))        # (N, 16) → (N, 16)
        x = self.fc3(x)                    # (N, 16) → (N, 3)
        return x

model2 = TwoHidden()
print(model2)

TwoHidden(
  (fc1): Linear(in_features=5, out_features=16, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=16, out_features=16, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=16, out_features=3, bias=True)
)


## 문제 3. 두 모델의 구조 비교표 (30점)
아래 항목으로 표를 만들어 출력하세요.

| 항목 | 은닉층 1개 | 은닉층 2개 |
| --- | --- | --- |
| Linear 층 수 | | |
| 전체 파라미터 수 | | |
| 활성화 함수 개수 | | |
| 출력 shape | | |


In [4]:
# 답안 — pd.DataFrame 으로 만들어 출력하세요
import pandas as pd

def summarize(model):
    n_linear = sum(isinstance(m, nn.Linear) for m in model.modules())
    n_params = sum(p.numel() for p in model.parameters())
    n_act = sum(isinstance(m, nn.ReLU) for m in model.modules())
    out_shape = tuple(model(torch.randn(8, 5)).shape)
    return [n_linear, n_params, n_act, out_shape]

table = pd.DataFrame(
    {"은닉층 1개": summarize(model1), "은닉층 2개": summarize(model2)},
    index=["Linear 층 수", "전체 파라미터 수", "활성화 함수 개수", "출력 shape (입력 8개 기준)"]
)
table.index.name = "항목"
print(table)

                     은닉층 1개  은닉층 2개
항목                                 
Linear 층 수                2       3
전체 파라미터 수               147     419
활성화 함수 개수                 1       2
출력 shape (입력 8개 기준)  (8, 3)  (8, 3)


## 문제 4. 구조 차이 설명 (20점)
아래 세 가지를 각각 두세 줄로 설명하세요.

1. 층을 늘리면 파라미터 수가 왜 늘어나는가
2. 활성화 함수가 없으면 층을 쌓아도 소용없는 이유
3. 층을 늘리는 것이 항상 좋은가 (근거를 함께)


**답:**

1.층을 늘리면 파라미터 수가 늘어나는 이유: Linear 층 하나는 (입력 수 × 출력 수)개의 가중치와 출력 수만큼의 편향을 갖습니다. 층을 하나 추가하면 그 층의 가중치 행렬과 편향이 통째로 더해지므로, 여기서는 16×16+16 = 272개가 늘어 147 → 419가 되었습니다.



2.활성화 함수가 없으면 층을 쌓아도 소용없는 이유: 선형 변환을 여러 번 합성해도 결과는 하나의 선형 변환입니다. W₂(W₁x + b₁) + b₂ = (W₂W₁)x + (W₂b₁ + b₂) 처럼 한 층짜리 모델로 정리되기 때문에, ReLU 같은 비선형 함수가 있어야 층마다 새로운 표현을 배울 수 있습니다.

3.층을 늘리는 것이 항상 좋은가: 아닙니다. 층이 많으면 더 복잡한 관계를 표현할 수 있지만, 파라미터가 늘어 데이터가 적을 때 과적합되기 쉽고, 학습 시간과 계산량이 늘며, 기울기 소실처럼 학습이 불안정해지는 문제도 생깁니다. 그래서 데이터 크기와 문제 난이도에 맞는 깊이를 검증 데이터로 비교해 고르는 것이 좋습니다.


## 문제 5. 임의 입력으로 확인 (보너스)
두 모델에 각각 `torch.randn(8, 5)` 를 넣어 출력 shape을 확인하고, 중간 층의 shape 변화를 출력하세요.


In [6]:
# 답안

x = torch.randn(8, 5)

def trace_shapes(model, x, name):
    print(f"[{name}] 입력: {tuple(x.shape)}")
    for layer_name, layer in model.named_children():
        x = layer(x)
        print(f"  {layer_name:6s} → {tuple(x.shape)}")
    print(f"  최종 출력: {tuple(x.shape)}\n")

trace_shapes(model1, x, "은닉층 1개")
trace_shapes(model2, x, "은닉층 2개")

[은닉층 1개] 입력: (8, 5)
  fc1    → (8, 16)
  relu   → (8, 16)
  fc2    → (8, 3)
  최종 출력: (8, 3)

[은닉층 2개] 입력: (8, 5)
  fc1    → (8, 16)
  relu1  → (8, 16)
  fc2    → (8, 16)
  relu2  → (8, 16)
  fc3    → (8, 3)
  최종 출력: (8, 3)

